# Notebook 05 — Open-Loop vs Closed-Loop Prediction

This notebook provides a side-by-side comparison of the two prediction
modes available in `chemical_reservoir`:

| Mode | Method | Test-phase input | Use case |
|------|--------|-----------------|----------|
| **Open-loop** | `solve_ode()` | Predetermined encoding signal | Simple, fast; periodic systems |
| **Closed-loop** | `solve_ode_closed_loop()` | Encoding + output feedback | Autonomous tracking; chaotic systems |

**Contents:**
1. Conceptual overview
2. Setup & imports
3. Van der Pol comparison
4. Brusselator comparison
5. Lotka–Volterra comparison
6. Lorenz comparison (chaotic)
7. Phase portraits
8. Summary table
9. Training efficiency — R² vs training size
10. Lorenz deep-dive — how much training is enough?
11. Multi-subnetwork architecture (leakage + fuel feed-in)
12. Scaling study — more subnetworks vs greater complexity
13. PDE benchmark — network architecture comparison
14. Discussion

---
## 5.1  Conceptual Overview

### Open-loop prediction

In open-loop mode, the reservoir is driven by a predetermined input
encoding signal $u(t)$ over the **full** time domain (training + test).
The readout weights $W_{\text{out}}$ are fitted via ridge regression on
the training data, and predictions are produced in a single forward pass.
The reservoir never sees its own output — it has no self-correction
mechanism.

**Strengths:** simple, fast, deterministic.  
**Weakness:** errors accumulate monotonically in the test region because
the reservoir has no feedback to counteract drift.

### Closed-loop prediction

In closed-loop mode, the **training** phase is identical (teacher
forcing).  During **testing**, each predicted output
$\hat{\mathbf{y}}(t_k)$ is fed back to modulate the encoding signal:

$$
\text{input}(t) = u(t) \times \bigl(1 + \alpha\,\hat{y}_{\text{norm}}(t_k)\bigr)
$$

where $\alpha$ is the feedback scale and $\hat{y}_{\text{norm}}$ is the
prediction normalized to zero mean / unit variance (using training statistics).

The encoding signal continues into the test region (so the reservoir
stays in a consistent input regime), while the feedback nudges it toward
the predicted attractor.

**Strengths:** self-correcting; tracks attractors autonomously.  
**Weakness:** slower (step-by-step integration); feedback can introduce
instabilities if $\alpha$ is too large.

---
## 5.2  Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import chemical_reservoir as cr
from chemical_reservoir.io import load_reservoir_from_json

NETWORK = Path('..') / 'network_configs' / 'high_dimensional_random.json'
N_TRAIN = 1600
N_TEST  = 800

print(f'Network:  {NETWORK.name}')
print(f'N_train:  {N_TRAIN}')
print(f'N_test:   {N_TEST}')

In [ ]:
def run_comparison(problem, n_train=N_TRAIN, n_test=N_TEST,
                   feedback_scale=0.5):
    """Run both open-loop and closed-loop on the same problem.
    
    Returns dict with keys 'open' and 'closed', each containing
    (results, metrics).
    """
    out = {}
    for mode in ('open', 'closed'):
        reservoir, _ = load_reservoir_from_json(str(NETWORK))
        reservoir.feature_expansion = 'delayed'
        
        if mode == 'open':
            results = reservoir.solve_ode(
                problem, n_train=n_train, n_test=n_test, encoding='mixed')
        else:
            results = reservoir.solve_ode_closed_loop(
                problem, n_train=n_train, n_test=n_test,
                encoding='mixed', feedback_scale=feedback_scale)
        
        metrics = cr.MetricsCalculator.calculate_all(
            y_true=results['y_true'], y_pred=results['y_pred'],
            t=results['t_all'], n_train=len(results['t_train']),
            variable_names=problem.variable_names)
        out[mode] = (results, metrics)
    
    return out


def plot_comparison(comp, problem, figsize=(14, 4)):
    """Plot open-loop vs closed-loop predictions side by side."""
    n_vars = len(problem.variable_names)
    fig, axes = plt.subplots(n_vars, 2, figsize=(figsize[0], figsize[1] * n_vars),
                             sharex='col', sharey='row')
    if n_vars == 1:
        axes = axes.reshape(1, -1)
    
    for col, (mode, label) in enumerate([('open', 'Open-loop'),
                                          ('closed', 'Closed-loop')]):
        results, metrics = comp[mode]
        t_all = results['t_all']
        n_train = len(results['t_train'])
        
        for row, name in enumerate(problem.variable_names):
            ax = axes[row, col]
            ax.plot(t_all, results['y_true'][:, row], 'b-',
                    lw=1.5, label='True', alpha=0.8)
            ax.plot(t_all, results['y_pred'][:, row], 'r--',
                    lw=1.5, label='Predicted', alpha=0.8)
            ax.axvline(t_all[n_train - 1], color='k', ls=':',
                       lw=1, alpha=0.5)
            ax.set_ylabel(name)
            if row == 0:
                ax.set_title(f'{label}  (R²={metrics.test_r2:.3f})')
            ax.legend(loc='upper right', fontsize=7)
        axes[-1, col].set_xlabel('t')
    
    fig.suptitle(f'{problem.name}: Open-loop vs Closed-loop', fontsize=13)
    plt.tight_layout()
    return fig

---
## 5.3  Van der Pol Oscillator

A relaxation oscillator with a stable limit cycle — periodic dynamics
are a good match for both modes.

In [ ]:
vdp = cr.problems.create_vanderpol_problem()
comp_vdp = run_comparison(vdp)

for mode in ('open', 'closed'):
    _, m = comp_vdp[mode]
    print(f'{mode:>7s}-loop:  Test R²={m.test_r2:.4f}  '
          f'RMSE={m.test_rmse:.4e}  MAE={m.test_mae:.4e}')

fig = plot_comparison(comp_vdp, vdp)
plt.show()

---
## 5.4  Brusselator

An autocatalytic chemical oscillator producing limit-cycle oscillations.

In [ ]:
bruss = cr.problems.create_brusselator_problem()
comp_bruss = run_comparison(bruss)

for mode in ('open', 'closed'):
    _, m = comp_bruss[mode]
    print(f'{mode:>7s}-loop:  Test R²={m.test_r2:.4f}  '
          f'RMSE={m.test_rmse:.4e}  MAE={m.test_mae:.4e}')

fig = plot_comparison(comp_bruss, bruss)
plt.show()

---
## 5.5  Lotka–Volterra Predator–Prey

Conservative oscillations — non-dissipative, so phase errors accumulate
more easily than in limit-cycle systems.

In [ ]:
lv = cr.problems.create_lotka_volterra_problem()
comp_lv = run_comparison(lv)

for mode in ('open', 'closed'):
    _, m = comp_lv[mode]
    print(f'{mode:>7s}-loop:  Test R²={m.test_r2:.4f}  '
          f'RMSE={m.test_rmse:.4e}  MAE={m.test_mae:.4e}')

fig = plot_comparison(comp_lv, lv)
plt.show()

---
## 5.6  Lorenz System (Chaotic)

The key stress test.  The Lorenz system has a positive Lyapunov exponent
($\lambda_1 \approx 0.9$), so prediction errors grow exponentially with
a timescale of ~1.1 time units.  Open-loop prediction diverges quickly;
closed-loop feedback extends the useful prediction horizon.

In [ ]:
lorenz = cr.problems.create_lorenz_problem()
comp_lorenz = run_comparison(lorenz, n_test=400)

for mode in ('open', 'closed'):
    _, m = comp_lorenz[mode]
    print(f'{mode:>7s}-loop:  Test R²={m.test_r2:.4f}  '
          f'RMSE={m.test_rmse:.4e}  MAE={m.test_mae:.4e}')

fig = plot_comparison(comp_lorenz, lorenz)
plt.show()

---
## 5.7  Phase Portraits

Compare the phase-space trajectories for Van der Pol and Lorenz to
visualise how well each mode tracks the attractor.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for col, (mode, label) in enumerate([('open', 'Open-loop'),
                                      ('closed', 'Closed-loop')]):
    # Van der Pol
    r, m = comp_vdp[mode]
    ax = axes[0, col]
    ax.plot(r['y_true'][:, 0], r['y_true'][:, 1], 'b-', lw=1, alpha=0.6, label='True')
    ax.plot(r['y_pred'][:, 0], r['y_pred'][:, 1], 'r--', lw=1, alpha=0.6, label='Predicted')
    ax.set_title(f'Van der Pol — {label} (R²={m.test_r2:.2f})')
    ax.set(xlabel='x', ylabel='v')
    ax.legend(fontsize=8)
    
    # Lorenz (x vs z)
    r, m = comp_lorenz[mode]
    ax = axes[1, col]
    ax.plot(r['y_true'][:, 0], r['y_true'][:, 2], 'b-', lw=0.8, alpha=0.6, label='True')
    ax.plot(r['y_pred'][:, 0], r['y_pred'][:, 2], 'r--', lw=0.8, alpha=0.6, label='Predicted')
    ax.set_title(f'Lorenz — {label} (R²={m.test_r2:.2f})')
    ax.set(xlabel='x', ylabel='z')
    ax.legend(fontsize=8)

plt.suptitle('Phase Portraits: Open-loop vs Closed-loop', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5.8  Summary Table

In [ ]:
import pandas as pd

rows = []
for name, comp in [('Van der Pol', comp_vdp),
                    ('Brusselator', comp_bruss),
                    ('Lotka-Volterra', comp_lv),
                    ('Lorenz', comp_lorenz)]:
    r2_open = comp['open'][1].test_r2
    r2_closed = comp['closed'][1].test_r2
    rmse_open = comp['open'][1].test_rmse
    rmse_closed = comp['closed'][1].test_rmse
    rows.append({
        'Problem': name,
        'Open-loop R²': f'{r2_open:.4f}',
        'Closed-loop R²': f'{r2_closed:.4f}',
        'Open-loop RMSE': f'{rmse_open:.4e}',
        'Closed-loop RMSE': f'{rmse_closed:.4e}',
        'R² change': f'{r2_closed - r2_open:+.4f}',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

---
## 5.9  Training Efficiency — R² vs Training Size

A key practical question is whether one mode reaches a target accuracy with
**less training data**.  Below we sweep `N_TRAIN` from 200 to 3200 while
keeping `N_TEST = 400` fixed, and plot the learning curve (test R²) for
both open-loop and closed-loop on each benchmark.

In [ ]:
train_sizes = [200, 400, 800, 1200, 1600, 2400, 3200]
n_test_fixed = 400

problems = [
    ('Van der Pol', cr.problems.create_vanderpol_problem()),
    ('Brusselator', cr.problems.create_brusselator_problem()),
    ('Lotka-Volterra', cr.problems.create_lotka_volterra_problem()),
    ('Lorenz', cr.problems.create_lorenz_problem()),
]

# Collect learning curves
curves = {}
for pname, prob in problems:
    curves[pname] = {'open': [], 'closed': []}
    for nt in train_sizes:
        print(f'  {pname:16s}  N_train={nt:4d} ...', end='', flush=True)
        comp = run_comparison(prob, n_train=nt, n_test=n_test_fixed)
        r2_o = comp['open'][1].test_r2
        r2_c = comp['closed'][1].test_r2
        curves[pname]['open'].append(r2_o)
        curves[pname]['closed'].append(r2_c)
        print(f'  open={r2_o:+.3f}  closed={r2_c:+.3f}')

print('\nDone.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
axes = axes.ravel()

for idx, (pname, _) in enumerate(problems):
    ax = axes[idx]
    ax.plot(train_sizes, curves[pname]['open'],  'o-', color='tab:blue',
            label='Open-loop', lw=2, markersize=5)
    ax.plot(train_sizes, curves[pname]['closed'], 's--', color='tab:red',
            label='Closed-loop', lw=2, markersize=5)
    ax.set_title(pname, fontsize=12)
    ax.set_ylabel('Test R²')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='grey', ls=':', lw=0.8)

axes[2].set_xlabel('N_train')
axes[3].set_xlabel('N_train')

fig.suptitle('Training Efficiency: Test R² vs Training Size', fontsize=14)
plt.tight_layout()
plt.show()

---
## 5.10  Lorenz Deep-Dive: How Much Training Is Enough?

The Lorenz system has a characteristic Lyapunov time $\tau_\lambda \approx 1.1$ time units.
Here we systematically increase the training duration and visualise both the
**time-series predictions** and the **3-D phase portraits** for both modes.

We use `N_TEST = 400` (fixed) and sweep `N_TRAIN` over a wide range so the
training window spans from ~2 to ~30+ Lyapunov times.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

lorenz_prob = cr.problems.create_lorenz_problem()
n_test_lorenz = 400
lorenz_train_sizes = [400, 800, 1600, 3200, 6400, 12800]

# Run all training sizes, store results for both modes
lorenz_runs = {}
for nt in lorenz_train_sizes:
    print(f'N_train={nt:5d} ...', end='', flush=True)
    comp = run_comparison(lorenz_prob, n_train=nt, n_test=n_test_lorenz)
    lorenz_runs[nt] = comp
    r2_o = comp['open'][1].test_r2
    r2_c = comp['closed'][1].test_r2
    print(f'  open R²={r2_o:+.3f}   closed R²={r2_c:+.3f}')

print('\nDone.')

In [ ]:
# Time-series: x(t) for each training size, open vs closed side by side
n_rows = len(lorenz_train_sizes)
fig, axes = plt.subplots(n_rows, 2, figsize=(14, 2.5 * n_rows), sharex='col')

for i, nt in enumerate(lorenz_train_sizes):
    comp = lorenz_runs[nt]
    for col, (mode, label) in enumerate([('open', 'Open-loop'),
                                          ('closed', 'Closed-loop')]):
        results, metrics = comp[mode]
        t_all = results['t_all']
        n_train = len(results['t_train'])
        ax = axes[i, col]
        
        # Plot x-component only for clarity
        ax.plot(t_all, results['y_true'][:, 0], 'b-', lw=1, alpha=0.8, label='True')
        ax.plot(t_all, results['y_pred'][:, 0], 'r-', lw=1, alpha=0.7, label='Predicted')
        ax.axvline(t_all[n_train - 1], color='k', ls=':', lw=1, alpha=0.6)
        
        if i == 0:
            ax.set_title(f'{label}', fontsize=11)
        ax.set_ylabel(f'N={nt}\nR²={metrics.test_r2:.2f}', fontsize=9)
        if i == n_rows - 1:
            ax.set_xlabel('t')
        if i == 0 and col == 0:
            ax.legend(fontsize=8, loc='upper right')

fig.suptitle('Lorenz x(t): Effect of Training Length on Prediction', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 3-D phase portraits: true vs predicted attractor for each training size
fig = plt.figure(figsize=(16, 4 * len(lorenz_train_sizes)))

for i, nt in enumerate(lorenz_train_sizes):
    comp = lorenz_runs[nt]
    for col, (mode, label) in enumerate([('open', 'Open-loop'),
                                          ('closed', 'Closed-loop')]):
        results, metrics = comp[mode]
        n_train = len(results['t_train'])
        y_true_test = results['y_true'][n_train:, :]
        y_pred_test = results['y_pred'][n_train:, :]
        
        ax = fig.add_subplot(len(lorenz_train_sizes), 2,
                             i * 2 + col + 1, projection='3d')
        ax.plot(y_true_test[:, 0], y_true_test[:, 1], y_true_test[:, 2],
                'b-', lw=0.6, alpha=0.7, label='True')
        ax.plot(y_pred_test[:, 0], y_pred_test[:, 1], y_pred_test[:, 2],
                'r-', lw=0.6, alpha=0.7, label='Predicted')
        ax.set_xlabel('x', fontsize=8)
        ax.set_ylabel('y', fontsize=8)
        ax.set_zlabel('z', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.set_title(f'{label}, N_train={nt}  (R²={metrics.test_r2:.2f})',
                     fontsize=9)
        if i == 0 and col == 0:
            ax.legend(fontsize=7)

fig.suptitle('Lorenz Phase Portraits: True vs Predicted (Test Region Only)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Summary plot: R² vs N_train for Lorenz (both modes)
fig, ax = plt.subplots(figsize=(8, 4))

r2_open_vals = [lorenz_runs[nt]['open'][1].test_r2 for nt in lorenz_train_sizes]
r2_closed_vals = [lorenz_runs[nt]['closed'][1].test_r2 for nt in lorenz_train_sizes]

ax.plot(lorenz_train_sizes, r2_open_vals, 'o-', color='tab:blue',
        lw=2, markersize=7, label='Open-loop')
ax.plot(lorenz_train_sizes, r2_closed_vals, 's--', color='tab:red',
        lw=2, markersize=7, label='Closed-loop')
ax.axhline(0, color='grey', ls=':', lw=1)
ax.set_xlabel('N_train (training points)', fontsize=11)
ax.set_ylabel('Test R²', fontsize=11)
ax.set_title('Lorenz System: Test R² vs Training Length', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

# Annotate approximate Lyapunov times (dt ≈ 0.01 for default Lorenz)
ax2 = ax.twiny()
ax2.set_xscale('log')
ax2.set_xlim(ax.get_xlim())
lyap_times = [nt * 0.01 / 1.1 for nt in lorenz_train_sizes]
ax2.set_xticks(lorenz_train_sizes)
ax2.set_xticklabels([f'{lt:.1f}' for lt in lyap_times], fontsize=9)
ax2.set_xlabel('Training window (Lyapunov times, τ_λ ≈ 1.1)', fontsize=10)

plt.tight_layout()
plt.show()

# Print table
print(f'{"N_train":>8s}  {"~Lyap times":>11s}  {"Open R²":>8s}  {"Closed R²":>10s}')
print('-' * 45)
for nt, r2o, r2c in zip(lorenz_train_sizes, r2_open_vals, r2_closed_vals):
    lt = nt * 0.01 / 1.1
    print(f'{nt:8d}  {lt:11.1f}  {r2o:+8.3f}  {r2c:+10.3f}')

---
## 5.11  Multi-Subnetwork Architecture with Leakage and Fuel Feed-In

The previous sections used a single strongly-coupled network.  Here we test
a different architecture:

| Feature | Original network | Multi-subnet leaky |
|---------|------------------|--------------------|
| Subnetworks | 5 coupled Brusselators | 6 weakly-coupled Brusselators |
| Coupling | Moderate Y–X exchange | **Small leakage** (X,Y diffusion, rate ~0.005) |
| Global coupling | None | **Shared fuel species F** (constant feed-in) |
| Species | 20 | 25 (including fuel F) |
| Reactions | 50 | 78 |

**Hypothesis:** Weak leakage preserves each subnetwork's distinct oscillation
frequency while the shared fuel provides a global synchronization signal.
This richer spectral structure may improve reservoir capacity, especially for
complex dynamics like Lorenz.

In [ ]:
from pathlib import Path
from chemical_reservoir.io import load_reservoir_from_json
from mpl_toolkits.mplot3d import Axes3D

# Load both networks
NET_ORIGINAL = Path('..') / 'network_configs' / 'high_dimensional_random.json'
NET_MULTISUBNET = Path('..') / 'network_configs' / 'multi_subnet_leaky.json'

lorenz_test = cr.problems.create_lorenz_problem()

# Compare across training sizes for Lorenz
train_sizes_compare = [800, 1600, 3200, 6400]
n_test_compare = 400

results_orig = {'open': [], 'closed': []}
results_multi = {'open': [], 'closed': []}

print("=== Original network (5 Brusselators, strong coupling) ===")
for nt in train_sizes_compare:
    for mode in ('open', 'closed'):
        reservoir, _ = load_reservoir_from_json(str(NET_ORIGINAL))
        reservoir.feature_expansion = 'delayed'
        if mode == 'open':
            res = reservoir.solve_ode(lorenz_test, n_train=nt, n_test=n_test_compare, encoding='mixed')
        else:
            res = reservoir.solve_ode_closed_loop(lorenz_test, n_train=nt, n_test=n_test_compare,
                                                   encoding='mixed', feedback_scale=0.5)
        metrics = cr.MetricsCalculator.calculate_all(
            y_true=res['y_true'], y_pred=res['y_pred'],
            t=res['t_all'], n_train=len(res['t_train']),
            variable_names=lorenz_test.variable_names)
        results_orig[mode].append((nt, res, metrics))
    r2o = results_orig['open'][-1][2].test_r2
    r2c = results_orig['closed'][-1][2].test_r2
    print(f"  N_train={nt:5d}  open R²={r2o:+.3f}  closed R²={r2c:+.3f}")

print("\n=== Multi-subnet leaky (6 Brusselators, leakage + fuel) ===")
for nt in train_sizes_compare:
    for mode in ('open', 'closed'):
        reservoir, _ = load_reservoir_from_json(str(NET_MULTISUBNET))
        reservoir.feature_expansion = 'delayed'
        if mode == 'open':
            res = reservoir.solve_ode(lorenz_test, n_train=nt, n_test=n_test_compare, encoding='mixed')
        else:
            res = reservoir.solve_ode_closed_loop(lorenz_test, n_train=nt, n_test=n_test_compare,
                                                   encoding='mixed', feedback_scale=0.5)
        metrics = cr.MetricsCalculator.calculate_all(
            y_true=res['y_true'], y_pred=res['y_pred'],
            t=res['t_all'], n_train=len(res['t_train']),
            variable_names=lorenz_test.variable_names)
        results_multi[mode].append((nt, res, metrics))
    r2o = results_multi['open'][-1][2].test_r2
    r2c = results_multi['closed'][-1][2].test_r2
    print(f"  N_train={nt:5d}  open R²={r2o:+.3f}  closed R²={r2c:+.3f}")

print("\nDone.")

In [ ]:
# Time-series comparison: Original vs Multi-subnet for best training size
# Pick one training size to show detailed predictions
nt_show = 3200

fig, axes = plt.subplots(3, 4, figsize=(18, 9), sharex='col')
var_names = ['x', 'y', 'z']

configs = [
    ('Original\nOpen', results_orig, 'open'),
    ('Original\nClosed', results_orig, 'closed'),
    ('Multi-sub\nOpen', results_multi, 'open'),
    ('Multi-sub\nClosed', results_multi, 'closed'),
]

for col, (title, res_dict, mode) in enumerate(configs):
    # Find the entry with nt_show
    entry = [(nt, r, m) for nt, r, m in res_dict[mode] if nt == nt_show][0]
    nt, res, metrics = entry
    t_all = res['t_all']
    n_train = len(res['t_train'])
    
    for row in range(3):
        ax = axes[row, col]
        ax.plot(t_all, res['y_true'][:, row], 'b-', lw=1, alpha=0.8, label='True')
        ax.plot(t_all, res['y_pred'][:, row], 'r-', lw=0.8, alpha=0.7, label='Predicted')
        ax.axvline(t_all[n_train-1], color='k', ls=':', lw=0.8)
        if row == 0:
            ax.set_title(f'{title}\nR²={metrics.test_r2:.2f}', fontsize=10)
        if col == 0:
            ax.set_ylabel(var_names[row], fontsize=11)
        if row == 2:
            ax.set_xlabel('t')
        if row == 0 and col == 0:
            ax.legend(fontsize=7, loc='upper right')

fig.suptitle(f'Lorenz Predictions: Original vs Multi-Subnet Network (N_train={nt_show})',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 3-D phase portraits: all 4 configurations at N_train=3200
fig = plt.figure(figsize=(16, 12))

for col, (title, res_dict, mode) in enumerate(configs):
    entry = [(nt, r, m) for nt, r, m in res_dict[mode] if nt == nt_show][0]
    _, res, metrics = entry
    n_train = len(res['t_train'])
    y_true_test = res['y_true'][n_train:]
    y_pred_test = res['y_pred'][n_train:]
    
    ax = fig.add_subplot(2, 2, col + 1, projection='3d')
    ax.plot(y_true_test[:, 0], y_true_test[:, 1], y_true_test[:, 2],
            'b-', lw=0.6, alpha=0.7, label='True')
    ax.plot(y_pred_test[:, 0], y_pred_test[:, 1], y_pred_test[:, 2],
            'r-', lw=0.6, alpha=0.7, label='Predicted')
    ax.set_xlabel('x', fontsize=9)
    ax.set_ylabel('y', fontsize=9)
    ax.set_zlabel('z', fontsize=9)
    ax.set_title(f'{title.replace(chr(10), " ")}\nR²={metrics.test_r2:.2f}', fontsize=10)
    ax.tick_params(labelsize=7)
    if col == 0:
        ax.legend(fontsize=8)

fig.suptitle(f'Lorenz Phase Portraits: Network Comparison (N_train={nt_show})',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# R² vs training size: both networks, both modes
fig, ax = plt.subplots(figsize=(10, 5))

styles = {
    ('orig', 'open'):   ('o-',  'tab:blue',   'Original open-loop'),
    ('orig', 'closed'): ('o--', 'tab:cyan',   'Original closed-loop'),
    ('multi', 'open'):  ('s-',  'tab:red',    'Multi-subnet open-loop'),
    ('multi', 'closed'):('s--', 'tab:orange', 'Multi-subnet closed-loop'),
}

for (net, mode), (fmt, color, label) in styles.items():
    res_dict = results_orig if net == 'orig' else results_multi
    r2_vals = [m.test_r2 for (_, _, m) in res_dict[mode]]
    ax.plot(train_sizes_compare, r2_vals, fmt, color=color, lw=2,
            markersize=7, label=label)

ax.axhline(0, color='grey', ls=':', lw=1)
ax.set_xlabel('N_train', fontsize=11)
ax.set_ylabel('Test R²', fontsize=11)
ax.set_title('Lorenz: Network Architecture Comparison', fontsize=12)
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print table
print(f"{'N_train':>7s}  {'Orig Open':>10s}  {'Orig Closed':>12s}  {'Multi Open':>11s}  {'Multi Closed':>13s}")
print('-' * 60)
for i, nt in enumerate(train_sizes_compare):
    print(f"{nt:7d}  {results_orig['open'][i][2].test_r2:+10.3f}  "
          f"{results_orig['closed'][i][2].test_r2:+12.3f}  "
          f"{results_multi['open'][i][2].test_r2:+11.3f}  "
          f"{results_multi['closed'][i][2].test_r2:+13.3f}")

In [ ]:
# Compare multi-subnet network on all 4 benchmark problems
import pandas as pd

all_problems = [
    ('Van der Pol', cr.problems.create_vanderpol_problem()),
    ('Brusselator', cr.problems.create_brusselator_problem()),
    ('Lotka-Volterra', cr.problems.create_lotka_volterra_problem()),
    ('Lorenz', cr.problems.create_lorenz_problem()),
]

nt_bench = 1600
n_test_bench = 400
rows_bench = []

print(f"Benchmark comparison at N_train={nt_bench}, N_test={n_test_bench}")
print("=" * 70)
for pname, prob in all_problems:
    for net_name, net_path in [('Original', NET_ORIGINAL), ('Multi-subnet', NET_MULTISUBNET)]:
        for mode in ('open', 'closed'):
            reservoir, _ = load_reservoir_from_json(str(net_path))
            reservoir.feature_expansion = 'delayed'
            if mode == 'open':
                res = reservoir.solve_ode(prob, n_train=nt_bench, n_test=n_test_bench, encoding='mixed')
            else:
                res = reservoir.solve_ode_closed_loop(prob, n_train=nt_bench, n_test=n_test_bench,
                                                      encoding='mixed', feedback_scale=0.5)
            m = cr.MetricsCalculator.calculate_all(
                y_true=res['y_true'], y_pred=res['y_pred'],
                t=res['t_all'], n_train=len(res['t_train']),
                variable_names=prob.variable_names)
            rows_bench.append({
                'Problem': pname, 'Network': net_name,
                'Mode': mode, 'Test R²': m.test_r2, 'Test RMSE': m.test_rmse
            })
    r = rows_bench[-4:]
    print(f"\n{pname}:")
    for rr in r:
        print(f"  {rr['Network']:12s} {rr['Mode']:7s}  R²={rr['Test R²']:+.3f}  RMSE={rr['Test RMSE']:.3e}")

df_bench = pd.DataFrame(rows_bench)
print("\n\nFull comparison table:")
print(df_bench.pivot_table(index=['Problem', 'Mode'], columns='Network',
                            values='Test R²', aggfunc='first').to_string())

---
## 5.12  Scaling Study: More Subnetworks vs Greater Complexity

Two strategies to increase reservoir capacity:
1. **Width scaling** — add more subnetworks (n_sub = 4, 6, 8, 10, 12) with standard complexity
2. **Depth scaling** — increase per-subnetwork complexity (minimal → standard → high) at fixed n_sub=6

The "high" complexity variant adds an intermediate species Z per module with
extra reactions (X→Z→Y pathway, Z autocatalysis), giving richer internal dynamics.

In [ ]:
import sys
sys.path.insert(0, str(Path('..') / 'scripts'))
from gen_parameterized_network import generate_network, save_network

# --- Width scaling: n_sub varies, standard complexity ---
width_configs = [4, 6, 8, 10, 12]
# --- Depth scaling: complexity varies, n_sub=6 ---
depth_configs = ['minimal', 'standard', 'high']

lorenz_scale = cr.problems.create_lorenz_problem()
vdp_scale = cr.problems.create_vanderpol_problem()
nt_scale = 1600
n_test_scale = 400

# Width scaling on Lorenz
print("=== WIDTH SCALING (Lorenz, open-loop) ===")
width_results = {}
for n_sub in width_configs:
    net = generate_network(n_sub=n_sub, complexity='standard')
    net_path = save_network(net)
    reservoir_w, _ = load_reservoir_from_json(net_path)
    reservoir_w.feature_expansion = 'delayed'
    res = reservoir_w.solve_ode(lorenz_scale, n_train=nt_scale, n_test=n_test_scale, encoding='mixed')
    m_w = cr.MetricsCalculator.calculate_all(
        y_true=res['y_true'], y_pred=res['y_pred'],
        t=res['t_all'], n_train=len(res['t_train']),
        variable_names=lorenz_scale.variable_names)
    width_results[n_sub] = {'r2': m_w.test_r2, 'rmse': m_w.test_rmse,
                            'n_species': net['_meta']['n_species'],
                            'n_reactions': net['_meta']['n_reactions']}
    print(f"  n_sub={n_sub:2d}  ({net['_meta']['n_species']:2d} sp, {net['_meta']['n_reactions']:3d} rxn)"
          f"  R²={m_w.test_r2:+.3f}  RMSE={m_w.test_rmse:.3e}")

# Depth scaling on Lorenz
print("\n=== DEPTH SCALING (Lorenz, open-loop, n_sub=6) ===")
depth_results = {}
for cplx in depth_configs:
    net = generate_network(n_sub=6, complexity=cplx)
    net_path = save_network(net)
    reservoir_d, _ = load_reservoir_from_json(net_path)
    reservoir_d.feature_expansion = 'delayed'
    res = reservoir_d.solve_ode(lorenz_scale, n_train=nt_scale, n_test=n_test_scale, encoding='mixed')
    m_d = cr.MetricsCalculator.calculate_all(
        y_true=res['y_true'], y_pred=res['y_pred'],
        t=res['t_all'], n_train=len(res['t_train']),
        variable_names=lorenz_scale.variable_names)
    depth_results[cplx] = {'r2': m_d.test_r2, 'rmse': m_d.test_rmse,
                           'n_species': net['_meta']['n_species'],
                           'n_reactions': net['_meta']['n_reactions']}
    print(f"  complexity={cplx:10s}  ({net['_meta']['n_species']:2d} sp, {net['_meta']['n_reactions']:3d} rxn)"
          f"  R²={m_d.test_r2:+.3f}  RMSE={m_d.test_rmse:.3e}")

# Also test the big one (n_sub=10, high)
print("\n=== LARGEST NETWORK (n_sub=10, high complexity) ===")
net_big = generate_network(n_sub=10, complexity='high')
net_big_path = save_network(net_big)
reservoir_big, _ = load_reservoir_from_json(net_big_path)
reservoir_big.feature_expansion = 'delayed'
res_big = reservoir_big.solve_ode(lorenz_scale, n_train=nt_scale, n_test=n_test_scale, encoding='mixed')
m_big = cr.MetricsCalculator.calculate_all(
    y_true=res_big['y_true'], y_pred=res_big['y_pred'],
    t=res_big['t_all'], n_train=len(res_big['t_train']),
    variable_names=lorenz_scale.variable_names)
print(f"  n_sub=10, high  ({net_big['_meta']['n_species']} sp, {net_big['_meta']['n_reactions']} rxn)"
      f"  R²={m_big.test_r2:+.3f}  RMSE={m_big.test_rmse:.3e}")

In [ ]:
# Plot scaling results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Width scaling
ax = axes[0]
n_subs = list(width_results.keys())
r2_width = [width_results[n]['r2'] for n in n_subs]
n_sp = [width_results[n]['n_species'] for n in n_subs]
ax.plot(n_subs, r2_width, 'o-', color='tab:blue', lw=2, markersize=8)
ax.set_xlabel('Number of subnetworks', fontsize=11)
ax.set_ylabel('Test R² (Lorenz)', fontsize=11)
ax.set_title('Width Scaling\n(standard complexity)', fontsize=12)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='grey', ls=':', lw=1)
# Annotate species count
for n, r2, sp in zip(n_subs, r2_width, n_sp):
    ax.annotate(f'{sp} sp', (n, r2), textcoords='offset points',
                xytext=(0, 10), fontsize=8, ha='center', color='grey')

# Depth scaling
ax = axes[1]
cplx_labels = list(depth_results.keys())
r2_depth = [depth_results[c]['r2'] for c in cplx_labels]
n_sp_d = [depth_results[c]['n_species'] for c in cplx_labels]
bars = ax.bar(cplx_labels, r2_depth, color=['tab:green', 'tab:blue', 'tab:red'], alpha=0.7)
ax.set_xlabel('Per-subnetwork complexity', fontsize=11)
ax.set_ylabel('Test R² (Lorenz)', fontsize=11)
ax.set_title('Depth Scaling\n(n_sub=6 fixed)', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='grey', ls=':', lw=1)
for bar, sp in zip(bars, n_sp_d):
    ax.annotate(f'{sp} sp', (bar.get_x() + bar.get_width()/2, bar.get_height()),
                textcoords='offset points', xytext=(0, 5), fontsize=9, ha='center')

# Add big network result
ax.axhline(m_big.test_r2, color='purple', ls='--', lw=1.5,
           label=f'n=10 high ({net_big["_meta"]["n_species"]} sp): R²={m_big.test_r2:.2f}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 5.13  PDE Benchmark: Network Architecture Comparison

We now apply the multi-subnet architecture to the PDE problems (Heat equation,
Advection-diffusion, Fisher equation, Burgers equation) and compare against
the original strongly-coupled network.

PDE solving uses `solve_pde()` (open-loop only — spatial prediction at each
time step).

In [ ]:
# PDE problems
pde_problems = [
    ('Heat Equation', cr.problems.create_heat_equation_problem()),
    ('Advection-Diffusion', cr.problems.create_advection_diffusion_problem()),
    ('Fisher Equation', cr.problems.create_fisher_equation_problem()),
    ('Burgers Equation', cr.problems.create_burgers_equation_problem()),
]

# Networks to compare
networks_pde = {
    'Original (20 sp)': str(NET_ORIGINAL),
    'Multi-sub 6 std (25 sp)': str(NET_MULTISUBNET),
    'Multi-sub 6 high (31 sp)': str(Path('..') / 'network_configs' / 'multisubnet_n6_high.json'),
    'Multi-sub 10 std (41 sp)': str(Path('..') / 'network_configs' / 'multisubnet_n10_standard.json'),
}

n_train_pde = 150
n_test_pde = 50

pde_results = []
print(f"PDE benchmark: n_train_time={n_train_pde}, n_test_time={n_test_pde}")
print("=" * 75)

for pname, prob in pde_problems:
    print(f"\n{pname}:")
    for net_label, net_path in networks_pde.items():
        reservoir_p, _ = load_reservoir_from_json(net_path)
        reservoir_p.feature_expansion = 'delayed'
        try:
            res_p = reservoir_p.solve_pde(prob, n_train_time=n_train_pde, n_test_time=n_test_pde)
            train_rmse = res_p['train_rmse']
            test_rmse = res_p['test_rmse']
            # Compute R² for test region
            sol_true_test = res_p['sol_true'][n_train_pde:]
            sol_pred_test = res_p['sol_pred'][n_train_pde:]
            ss_res = np.sum((sol_true_test - sol_pred_test)**2)
            ss_tot = np.sum((sol_true_test - np.mean(sol_true_test))**2)
            r2_pde = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        except Exception as e:
            train_rmse = test_rmse = r2_pde = float('nan')
            print(f"  ERROR: {e}")

        pde_results.append({
            'Problem': pname, 'Network': net_label,
            'Train RMSE': train_rmse, 'Test RMSE': test_rmse, 'Test R²': r2_pde
        })
        print(f"  {net_label:28s}  train={train_rmse:.4e}  test={test_rmse:.4e}  R²={r2_pde:+.4f}")

print("\nDone.")

In [ ]:
# PDE results visualization
df_pde = pd.DataFrame(pde_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test R² by problem and network
ax = axes[0]
pivot_r2 = df_pde.pivot(index='Problem', columns='Network', values='Test R²')
pivot_r2.plot(kind='bar', ax=ax, rot=20, width=0.7)
ax.set_ylabel('Test R²', fontsize=11)
ax.set_title('PDE Problems: Test R² by Network Architecture', fontsize=11)
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='grey', ls=':', lw=1)

# Test RMSE by problem and network
ax = axes[1]
pivot_rmse = df_pde.pivot(index='Problem', columns='Network', values='Test RMSE')
pivot_rmse.plot(kind='bar', ax=ax, rot=20, width=0.7)
ax.set_ylabel('Test RMSE', fontsize=11)
ax.set_title('PDE Problems: Test RMSE by Network Architecture', fontsize=11)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
ax.set_yscale('log')

plt.tight_layout()
plt.show()

# Summary table
print("\nPDE Comparison Table:")
print(df_pde.pivot_table(index='Problem', columns='Network',
                          values=['Test R²', 'Test RMSE']).to_string())

---
## 5.10  Discussion

> **Note:** Results vary between runs due to stochastic reservoir initialization
> (initial concentrations drawn from physical constraints).  Re-run the notebook
> to observe different realisations.

### When does closed-loop help?

- **Oscillatory systems with strong limit cycles** (Brusselator): closed-loop
  often yields the largest improvement.  The output feedback helps maintain
  phase coherence in the test region, keeping the prediction locked to the
  oscillation frequency.

- **Periodic systems** (Van der Pol): the benefit is problem-dependent and
  varies with initialization.  Some runs show improvement, others slight
  degradation.  On average across many seeds, closed-loop tends to be
  comparable or slightly better.

### Training efficiency

The learning curves (Section 5.9) reveal whether one mode reaches a target
accuracy with less training data.  Key observations:

- If closed-loop reaches a given R² at a smaller `N_TRAIN`, it is more
  **data-efficient** — the feedback provides implicit regularisation.
- If both curves plateau at similar `N_TRAIN`, the modes have similar
  sample complexity but may differ in final accuracy.
- The computational cost per training point is higher for closed-loop
  (step-by-step integration at test time), so wall-clock efficiency
  should also be considered.

### When might open-loop be preferred?

- **Speed**: open-loop makes predictions in a single forward pass, while
  closed-loop steps through the test region one time-point at a time
  (requiring an ODE integration at each step).

- **Stability**: if the feedback scale $\alpha$ is poorly tuned, the
  closed-loop can amplify errors rather than correct them — particularly
  visible in chaotic systems where small feedback perturbations grow
  exponentially.

- **Conservative systems** (Lotka–Volterra): the output feedback can
  slightly perturb the energy-conserving structure, occasionally leading
  to amplitude drift.  Open-loop is often more reliable here.

- **Chaotic systems** (Lorenz): both modes struggle because the underlying
  reservoir dynamics are driven by mass-action kinetics that may not have
  sufficient capacity to represent the chaotic attractor.  The feedback
  can either help or hurt depending on the initialization.

### Feedback scale selection

The feedback scale $\alpha$ controls how strongly the predicted output
modulates the encoding input.  Values of 0.3–0.7 generally work well.
Too small ($< 0.1$) and the feedback is negligible; too large ($> 1.0$)
and the feedback can destabilize the reservoir dynamics.

### Summary

Both modes are available via the API (`solve_ode` vs `solve_ode_closed_loop`)
and the CLI (`--mode open-loop` vs `--mode closed-loop`).  The best choice
depends on the problem:
- **Strongly periodic** systems → closed-loop tends to help
- **Conservative** or **chaotic** systems → open-loop may be more reliable
- **Quick experiments** → open-loop is faster
- Check the learning curves above to see which mode is more data-efficient
  for your problem of interest